<a href="https://colab.research.google.com/github/kuruvajayanth12/Neural-Networks-and-Deep-Learning/blob/main/EXP_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 !pip install ultralytics opencv-python


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving park_video.mp4 to park_video (1).mp4


In [ ]:
import cv2
from ultralytics import YOLO
from IPython.display import HTML
from base64 import b64encode

# Load model
model = YOLO("yolov8n.pt")

# Get uploaded file name
video_path = list(uploaded.keys())[0]

cap = cv2.VideoCapture(video_path)

# Output video
out = cv2.VideoWriter(
    "output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    20,
    (int(cap.get(3)), int(cap.get(4)))
)

VEHICLE_CLASSES = [2, 3, 5, 7]
LINE_Y = 300

previous_positions = {}
entered = set()
exited = set()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True, verbose=False)

    for r in results:
        boxes = r.boxes

        if boxes.id is None:
            continue

        for i, box in enumerate(boxes.xyxy):
            obj_id = int(boxes.id[i])
            cls = int(boxes.cls[i])

            if cls not in VEHICLE_CLASSES:
                continue

            x1, y1, x2, y2 = map(int, box)
            center_y = (y1 + y2) // 2

            # Entry/Exit logic
            if obj_id in previous_positions:
                prev_y = previous_positions[obj_id]

                if prev_y < LINE_Y and center_y >= LINE_Y:
                    if obj_id not in entered:
                        entered.add(obj_id)

                elif prev_y > LINE_Y and center_y <= LINE_Y:
                    if obj_id not in exited:
                         exited.add(obj_id)

            previous_positions[obj_id] = center_y

            # Draw box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

    # Draw line
    cv2.line(frame, (0, LINE_Y), (frame.shape[1], LINE_Y), (0,0,255), 2)

    # Show counts
    cv2.putText(frame, f"Entered: {len(entered)}", (10,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 2)

    cv2.putText(frame, f"Exited: {len(exited)}", (10,80),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)

    out.write(frame)

cap.release()
out.release()

In [ ]:
!apt-get install ffmpeg -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


In [ ]:
!ffmpeg -i output.mp4 -vcodec libx264 output_fixed.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open('output_fixed.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=600 controls>
  <source src="{data_url}" type="video/mp4">
</video>
""")